# NASS raw data exploration

这个 notebook 用同一套方法分析 `annual_supply_raw.csv` 和 `stocks_raw.csv`：

- 每列的数据类型、缺失率、唯一值数量、样例值、数值范围
- 关键分类列有哪些取值
- `Value` 字段在不同业务维度下的范围
- stocks 额外结合 clean/mart/WASDE/supply 数据构建指标


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

ROOT = Path(".")
RAW_NASS_DIR = ROOT / "data" / "raw" / "nass"
CLEAN_DIR = ROOT / "data" / "clean"
MART_DIR = ROOT / "data" / "mart"

annual_supply_raw_path = RAW_NASS_DIR / "annual_supply_raw.csv"
stocks_raw_path = RAW_NASS_DIR / "stocks_raw.csv"

nass_stocks_clean_path = CLEAN_DIR / "nass_stocks.csv"
stocks_mart_path = MART_DIR / "mart_stocks_quarterly.csv"
supply_mart_path = MART_DIR / "mart_supply_annual.csv"
wasde_balance_path = MART_DIR / "mart_wasde_balance.csv"


In [ ]:
def clean_quickstats_value(value_series: pd.Series) -> pd.Series:
    """Convert NASS QuickStats value strings such as '9,024,445,000' to numeric."""
    cleaned = (
        value_series
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace(r"[^0-9.\-]", "", regex=True)
    )
    cleaned = cleaned.mask(cleaned.eq(""))
    return pd.to_numeric(cleaned, errors="coerce")


def load_quickstats_raw(path: str | Path) -> pd.DataFrame:
    """Load a NASS raw CSV and add standardized analysis columns."""
    df = pd.read_csv(path, low_memory=False)
    df = df.rename(columns={"Value": "value_raw", "CV (%)": "cv_pct"})

    if "value_raw" in df.columns:
        df["value_num"] = clean_quickstats_value(df["value_raw"])
    if "cv_pct" in df.columns:
        df["cv_pct"] = pd.to_numeric(df["cv_pct"], errors="coerce")
    if "load_time" in df.columns:
        df["load_time_dt"] = pd.to_datetime(df["load_time"], errors="coerce")

    return df


def unique_preview(series: pd.Series, limit: int = 8) -> str:
    values = sorted(series.dropna().astype(str).unique())
    suffix = "" if len(values) <= limit else f" ... (+{len(values) - limit})"
    return ", ".join(values[:limit]) + suffix


def column_profile(df: pd.DataFrame, sample_size: int = 8) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        row = {
            "column": col,
            "dtype": str(s.dtype),
            "non_null": int(s.notna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "n_unique": int(s.nunique(dropna=True)),
            "sample_values": s.dropna().astype(str).unique()[:sample_size].tolist(),
        }
        if pd.api.types.is_numeric_dtype(s):
            row.update({"min": s.min(), "median": s.median(), "max": s.max()})
        else:
            row.update({"min": None, "median": None, "max": None})
        rows.append(row)
    return pd.DataFrame(rows)


def show_value_counts(df: pd.DataFrame, columns: list[str], top_n: int = 30) -> None:
    for col in columns:
        if col not in df.columns:
            print(f"\n## {col} not found")
            continue
        print(f"\n## {col} | unique={df[col].nunique(dropna=True)}")
        display(
            df[col]
            .value_counts(dropna=False)
            .head(top_n)
            .rename_axis(col)
            .reset_index(name="rows")
        )


def value_range_summary(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    group_cols = [col for col in group_cols if col in df.columns]
    return (
        df.groupby(group_cols, dropna=False)
        .agg(
            rows=("value_num", "size"),
            non_null=("value_num", "count"),
            min_value=("value_num", "min"),
            median_value=("value_num", "median"),
            max_value=("value_num", "max"),
        )
        .reset_index()
        .sort_values(group_cols)
    )


def short_desc_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["short_desc", "unit_desc"], dropna=False)
        .agg(
            rows=("value_num", "size"),
            years=("year", lambda x: f"{int(x.min())}-{int(x.max())}" if x.notna().any() else None),
            states=("state_alpha", "nunique"),
            reference_periods=("reference_period_desc", unique_preview),
            min_value=("value_num", "min"),
            median_value=("value_num", "median"),
            max_value=("value_num", "max"),
        )
        .reset_index()
        .sort_values("short_desc")
    )


## annual_supply_raw.csv

这个文件主要回答：面积、单产、产量分别是什么，按作物、年份、地区、报告口径怎么分布。


In [ ]:
annual_supply_raw = load_quickstats_raw(annual_supply_raw_path)

print("Shape:", annual_supply_raw.shape)
display(annual_supply_raw.head())
display(column_profile(annual_supply_raw))


In [ ]:
annual_supply_key_cols = [
    "commodity_desc",
    "statisticcat_desc",
    "unit_desc",
    "short_desc",
    "agg_level_desc",
    "state_alpha",
    "reference_period_desc",
    "year",
]

show_value_counts(annual_supply_raw, annual_supply_key_cols)


In [ ]:
annual_supply_value_summary = value_range_summary(
    annual_supply_raw,
    ["commodity_desc", "statisticcat_desc", "unit_desc", "agg_level_desc"],
)
display(annual_supply_value_summary)

display(short_desc_summary(annual_supply_raw))


In [ ]:
# Dashboard 常用口径：年度最终值 + NATIONAL/STATE
annual_supply_final = annual_supply_raw[
    annual_supply_raw["reference_period_desc"].eq("YEAR")
    & annual_supply_raw["agg_level_desc"].isin(["NATIONAL", "STATE"])
].copy()

annual_supply_pivot = (
    annual_supply_final
    .pivot_table(
        index=["commodity_desc", "year", "state_alpha", "state_name", "agg_level_desc"],
        columns="statisticcat_desc",
        values="value_num",
        aggfunc="first",
    )
    .reset_index()
)
annual_supply_pivot.columns.name = None
annual_supply_pivot = annual_supply_pivot.rename(
    columns={
        "AREA PLANTED": "area_planted_acres",
        "AREA HARVESTED": "area_harvested_acres",
        "YIELD": "yield_bu_per_acre",
        "PRODUCTION": "production_bu",
    }
)
annual_supply_pivot["area_planted_mil_acres"] = annual_supply_pivot["area_planted_acres"] / 1_000_000
annual_supply_pivot["area_harvested_mil_acres"] = annual_supply_pivot["area_harvested_acres"] / 1_000_000
annual_supply_pivot["production_mbu"] = annual_supply_pivot["production_bu"] / 1_000_000

display(annual_supply_pivot.head(20))


## stocks_raw.csv

Stocks 是 **POINT IN TIME** 数据，重点看季度时间点：

- `FIRST OF MAR`
- `FIRST OF JUN`
- `FIRST OF SEP`
- `FIRST OF DEC`

核心分析字段：`commodity_desc`、`reference_period_desc`、`stock_month`、`stock_date`、`agg_level_desc`、`state_alpha`、`value_num`、`stocks_mbu`。


In [ ]:
STOCK_MONTH_MAP = {
    "FIRST OF MAR": 3,
    "FIRST OF JUN": 6,
    "FIRST OF SEP": 9,
    "FIRST OF DEC": 12,
}


def add_stock_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["stock_month"] = out["reference_period_desc"].map(STOCK_MONTH_MAP)

    if "begin_code" in out.columns:
        begin_month = out["begin_code"].where(out["begin_code"].isin([3, 6, 9, 12]))
        out["stock_month"] = out["stock_month"].fillna(begin_month)

    month_text = out["stock_month"].astype("Int64").astype("string").str.zfill(2)
    out["stock_date"] = pd.to_datetime(
        out["year"].astype("Int64").astype("string") + "-" + month_text + "-01",
        errors="coerce",
    )

    is_stock_bushels = out["statisticcat_desc"].eq("STOCKS") & out["unit_desc"].eq("BU")
    out["stocks_bu"] = out["value_num"].where(is_stock_bushels)
    out["stocks_mbu"] = out["stocks_bu"] / 1_000_000
    return out


stocks_raw = add_stock_fields(load_quickstats_raw(stocks_raw_path))

print("Shape:", stocks_raw.shape)
display(stocks_raw.head())
display(column_profile(stocks_raw))


In [ ]:
stocks_key_cols = [
    "commodity_desc",
    "class_desc",
    "util_practice_desc",
    "statisticcat_desc",
    "unit_desc",
    "short_desc",
    "agg_level_desc",
    "state_alpha",
    "reference_period_desc",
    "stock_month",
    "year",
]

show_value_counts(stocks_raw, stocks_key_cols)


In [ ]:
stocks_value_summary = value_range_summary(
    stocks_raw,
    ["commodity_desc", "agg_level_desc", "reference_period_desc", "unit_desc"],
)
display(stocks_value_summary)

display(short_desc_summary(stocks_raw))

stocks_calendar_summary = (
    stocks_raw.groupby(["commodity_desc", "reference_period_desc", "stock_month"], dropna=False)
    .agg(
        rows=("stocks_mbu", "size"),
        years=("year", lambda x: f"{int(x.min())}-{int(x.max())}" if x.notna().any() else None),
        states=("state_alpha", "nunique"),
        min_stocks_mbu=("stocks_mbu", "min"),
        median_stocks_mbu=("stocks_mbu", "median"),
        max_stocks_mbu=("stocks_mbu", "max"),
    )
    .reset_index()
    .sort_values(["commodity_desc", "stock_month"])
)
display(stocks_calendar_summary)


In [ ]:
stocks_analysis = stocks_raw[
    stocks_raw["agg_level_desc"].isin(["NATIONAL", "STATE"])
    & stocks_raw["stock_date"].notna()
].copy()

stocks_analysis = stocks_analysis[
    [
        "commodity_desc",
        "year",
        "stock_date",
        "stock_month",
        "reference_period_desc",
        "agg_level_desc",
        "state_alpha",
        "state_name",
        "stocks_bu",
        "stocks_mbu",
        "load_time_dt",
    ]
].rename(
    columns={
        "commodity_desc": "commodity",
        "agg_level_desc": "agg_level",
        "load_time_dt": "load_time",
    }
)

stocks_analysis = stocks_analysis.sort_values(
    ["commodity", "agg_level", "state_alpha", "stock_date"]
)

display(stocks_analysis.head(20))


## stocks + clean/mart/WASDE/supply 联动分析

Stocks 单独看绝对值还不够，下面接入：

- `mart_stocks_quarterly.csv`: 已清洗季度库存
- `mart_supply_annual.csv`: 库存/产量比例
- `mart_wasde_balance.csv`: WASDE ending stocks 和 stocks-to-use


In [ ]:
clean_stocks = pd.read_csv(nass_stocks_clean_path, parse_dates=["load_time"])
mart_stocks = pd.read_csv(stocks_mart_path, parse_dates=["stock_date", "load_time"])
mart_supply = pd.read_csv(supply_mart_path, parse_dates=["latest_load_time"])
wasde_balance = pd.read_csv(wasde_balance_path)

sources_overview = pd.DataFrame(
    [
        {"dataset": "raw stocks", "path": str(stocks_raw_path), "rows": len(stocks_raw), "columns": stocks_raw.shape[1]},
        {"dataset": "clean stocks", "path": str(nass_stocks_clean_path), "rows": len(clean_stocks), "columns": clean_stocks.shape[1]},
        {"dataset": "mart stocks", "path": str(stocks_mart_path), "rows": len(mart_stocks), "columns": mart_stocks.shape[1]},
        {"dataset": "mart supply", "path": str(supply_mart_path), "rows": len(mart_supply), "columns": mart_supply.shape[1]},
        {"dataset": "mart WASDE", "path": str(wasde_balance_path), "rows": len(wasde_balance), "columns": wasde_balance.shape[1]},
    ]
)
display(sources_overview)
display(mart_stocks.head())
display(wasde_balance.head())


In [ ]:
# 1) 全国季度库存：YoY、5年均值、相对5年均值偏离
stocks_national = mart_stocks[mart_stocks["agg_level"].eq("NATIONAL")].copy()
stocks_national = stocks_national.sort_values(["commodity", "stock_month", "stock_date"])


def add_stock_benchmarks(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("stock_date").copy()
    group["yoy_change_mbu"] = group["stocks_mbu"].diff()
    group["yoy_change_pct"] = group["stocks_mbu"].pct_change() * 100
    group["five_year_avg_mbu"] = group["stocks_mbu"].shift(1).rolling(5, min_periods=3).mean()
    group["vs_5y_avg_mbu"] = group["stocks_mbu"] - group["five_year_avg_mbu"]
    group["vs_5y_avg_pct"] = group["vs_5y_avg_mbu"] / group["five_year_avg_mbu"] * 100
    return group


stocks_benchmark_parts = [
    add_stock_benchmarks(group)
    for _, group in stocks_national.groupby(["commodity", "stock_month"], sort=False)
]
stocks_benchmarks = pd.concat(stocks_benchmark_parts, ignore_index=True)

latest_stock_benchmarks = (
    stocks_benchmarks
    .sort_values("stock_date")
    .groupby(["commodity", "stock_month"], as_index=False)
    .tail(1)
    .sort_values(["commodity", "stock_month"])
)

display(
    latest_stock_benchmarks[
        [
            "commodity",
            "stock_date",
            "reference_period_desc",
            "stocks_mbu",
            "yoy_change_mbu",
            "yoy_change_pct",
            "five_year_avg_mbu",
            "vs_5y_avg_mbu",
            "vs_5y_avg_pct",
        ]
    ]
)


In [ ]:
# 2) 最新季度库存的州分布：哪些州贡献了全国库存？
stocks_state = mart_stocks[mart_stocks["agg_level"].eq("STATE")].copy()
latest_stock_dates = stocks_national.groupby(["commodity", "stock_month"], as_index=False)["stock_date"].max()

latest_us_stocks = stocks_national.merge(
    latest_stock_dates,
    on=["commodity", "stock_month", "stock_date"],
    how="inner",
)
latest_state_stocks = stocks_state.merge(
    latest_stock_dates,
    on=["commodity", "stock_month", "stock_date"],
    how="inner",
)

state_stock_share = latest_state_stocks.merge(
    latest_us_stocks[["commodity", "stock_month", "stock_date", "stocks_mbu"]]
    .rename(columns={"stocks_mbu": "us_stocks_mbu"}),
    on=["commodity", "stock_month", "stock_date"],
    how="left",
)
state_stock_share["share_of_us_pct"] = state_stock_share["stocks_mbu"] / state_stock_share["us_stocks_mbu"] * 100

top_state_stock_share = (
    state_stock_share
    .sort_values(["commodity", "stock_month", "share_of_us_pct"], ascending=[True, True, False])
    .groupby(["commodity", "stock_month"], as_index=False)
    .head(10)
)

display(
    top_state_stock_share[
        [
            "commodity",
            "stock_date",
            "reference_period_desc",
            "state_alpha",
            "state_name",
            "stocks_mbu",
            "us_stocks_mbu",
            "share_of_us_pct",
        ]
    ]
)


In [ ]:
# 3) 库存 / 产量：粗略衡量库存相对当年产量的比例。
# 注意：不同 stock_month 的经济含义不同，Sep 1 更接近上一市场年度 ending stocks。
supply_us = mart_supply[mart_supply["state_alpha"].eq("US")].copy()
supply_us = supply_us[
    [
        "commodity",
        "year",
        "planted_acres_mil",
        "harvested_acres_mil",
        "yield_bu_per_acre",
        "production_mbu",
    ]
]

stocks_vs_supply = stocks_national.merge(
    supply_us,
    on=["commodity", "year"],
    how="left",
)
stocks_vs_supply["stocks_to_production_pct"] = stocks_vs_supply["stocks_mbu"] / stocks_vs_supply["production_mbu"] * 100

latest_stocks_vs_supply = (
    stocks_vs_supply
    .sort_values("stock_date")
    .groupby(["commodity", "stock_month"], as_index=False)
    .tail(1)
    .sort_values(["commodity", "stock_month"])
)

display(
    latest_stocks_vs_supply[
        [
            "commodity",
            "stock_date",
            "reference_period_desc",
            "stocks_mbu",
            "production_mbu",
            "stocks_to_production_pct",
        ]
    ]
)


In [ ]:
# 4) Sep 1 NASS Grain Stocks vs WASDE ending stocks
# 对玉米/大豆来说，Sep 1 stocks 通常可以和上一市场年度 ending stocks 对照。
sep_stocks = stocks_national[stocks_national["stock_month"].eq(9)].copy()
sep_stocks["marketing_year"] = (
    (sep_stocks["year"] - 1).astype(str)
    + "/"
    + sep_stocks["year"].astype(str).str[-2:]
)

wasde_ending = wasde_balance[
    [
        "report_month",
        "commodity",
        "marketing_year",
        "estimate_type",
        "vintage",
        "ending_stocks",
        "use_total",
        "stocks_to_use_pct",
    ]
].copy()

sep_vs_wasde = sep_stocks.merge(
    wasde_ending,
    on=["commodity", "marketing_year"],
    how="inner",
)
sep_vs_wasde["nass_sep_minus_wasde_mbu"] = sep_vs_wasde["stocks_mbu"] - sep_vs_wasde["ending_stocks"]

display(
    sep_vs_wasde[
        [
            "commodity",
            "stock_date",
            "marketing_year",
            "stocks_mbu",
            "report_month",
            "estimate_type",
            "vintage",
            "ending_stocks",
            "nass_sep_minus_wasde_mbu",
            "use_total",
            "stocks_to_use_pct",
        ]
    ]
    .sort_values(["commodity", "stock_date", "marketing_year"])
    .tail(20)
)


## Stocks 分析时最值得关注的指标

- `stocks_mbu`: 百万蒲式耳库存。
- `yoy_change_mbu` / `yoy_change_pct`: 同一季度时间点的同比变化。
- `five_year_avg_mbu`: 同一季度时间点的过去 5 年均值。
- `vs_5y_avg_mbu` / `vs_5y_avg_pct`: 当前库存相对 5 年均值偏离。
- `share_of_us_pct`: 州库存占全国库存比例。
- `stocks_to_production_pct`: 库存相对产量比例。
- `stocks_to_use_pct`: WASDE 口径的库存消费比，更适合判断供需松紧。

注意：`FIRST OF MAR/JUN/SEP/DEC` 不应该简单混在一条同比序列里比较，最好按 `stock_month` 分组，因为每个季度点的季节性完全不同。
